# 06 · Hurst Exponent — Finance Concept

**Contexto:** Harold Hurst (1951) descubrió que las series naturales tienen memoria de largo plazo. Mandelbrot (1971) aplicó el exponente H a mercados financieros. El exponente H cuantifica cuánta memoria tiene una serie: H=0.5 ruido blanco, H>0.5 persistencia, H<0.5 anti-persistencia.

**Campo de origen:** Hidrología → Finanzas cuantitativas → Risk Management  
**Dataset:** Alicorp S.A.A. (ALICORC1) — Bolsa de Valores de Lima  
**Fuente real:** `yfinance` ticker `ALICORC1.LM` o portal BVL

---

## Marco teórico

### Exponente de Hurst H

$$E\left[\frac{R(n)}{S(n)}\right] \approx c \cdot n^H \implies H = \text{slope of } \log(R/S) \text{ vs. } \log(n)$$

| $H$ | Régimen | Autocorrelación |
|-----|---------|----------------|
| $H < 0.5$ | Anti-persistente | $\rho(k) < 0$ — reversión |
| $H = 0.5$ | Ruido blanco | $\rho(k) = 0$ — independiente |
| $H > 0.5$ | Persistente | $\rho(k) > 0$ — memoria larga |

### Corrección de Safety Stock por Hurst

$$SS_{Hurst} = z \cdot \sigma \cdot L^H \qquad \text{vs.} \qquad SS_{clásico} = z \cdot \sigma \cdot L^{0.5}$$

### Conexión con ARFIMA

$$d = H - 0.5 \implies ARFIMA(p,d,q) \text{ captura la memoria larga}$$

**Referencias:** Hurst (1951). *Trans. Am. Soc. Civ. Eng.* 116. Mandelbrot & Van Ness (1968). *SIAM Rev.* 10(4). Lo (1991). *Econometrica* 59(5).

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats as sp_stats
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    price='#1E293B', hurst='#DC2626', dfa='#2563EB',
    fill='#DBEAFE',  green='#15803D', neutral='#94A3B8',
    rs='#7C3AED',    bvl='#F59E0B',   half='#64748B'
)
np.random.seed(42)
print('OK')

In [ ]:
# ── DATOS — ALICORC1 BVL ──────────────────────────────────────────────────────
# Fuente real:
#   import yfinance as yf
#   raw = yf.download('ALICORC1.LM', start='2010-01-01', end='2024-12-31')
#   prices = raw['Close'].dropna()
#   returns = prices.pct_change().dropna()
#
# Portal BVL: https://www.bvl.com.pe/emisores/ALICORC1

try:
    import yfinance as yf
    raw = yf.download('ALICORC1.LM', start='2010-01-01', end='2024-12-31', progress=False)
    if len(raw) > 100:
        prices  = raw['Close'].squeeze().dropna()  # squeeze: DataFrame -> Series
        returns = prices.pct_change().dropna()
        SOURCE  = 'yfinance — ALICORC1.LM (BVL, datos reales)'
        print(f'Datos reales: {len(prices)} obs.')
    else:
        raise ValueError('insuficiente')
except Exception as e:
    print(f'yfinance no disponible — simulacion calibrada')
    SOURCE = 'Simulacion calibrada (estadisticos reales ALICORC1 2010-2024)'
    n   = 3500
    dates = pd.bdate_range('2010-01-04', periods=n)
    phi, sigma = 0.12, 0.0105
    rets = [0.0]
    for t in range(1, n):
        rets.append(phi*rets[-1] + np.random.normal(0, sigma))
    rets = np.array(rets)
    rets[200:220]  += np.random.normal(0.005, 0.015, 20)
    rets[1300:1600]+= np.random.normal(-0.001, 0.008, 300)
    rets[2500:2530]+= np.random.normal(-0.015, 0.025, 30)
    rets[2530:2600]+= np.random.normal(0.008, 0.018, 70)
    returns = pd.Series(rets, index=dates)
    prices  = pd.Series(5.50 * np.cumprod(1+rets), index=dates)

print(f'Fuente  : {SOURCE}')
print(f'Periodo : {returns.index[0].date()} -> {returns.index[-1].date()}')
print(f'n       : {len(returns)} dias habiles')
print(f'mu ret  : {returns.mean():.5f}  ({returns.mean()*252:.2%} anual)')
print(f'sigma   : {returns.std():.5f}  ({returns.std()*np.sqrt(252):.2%} anual)')

## Mini-EDA

In [ ]:
from statsmodels.tsa.stattools import acf
r = returns
acf_r2 = acf(r**2, nlags=40, fft=True)
ci_acf = 1.96 / np.sqrt(len(r))
sig_lags = [i for i in range(1,41) if abs(acf_r2[i]) > ci_acf]

print(f'Estadisticos clave:')
print(f'  mu diario : {r.mean():.5f}  ({r.mean()*252:.2%} anual)')
print(f'  sigma     : {r.std():.5f}  ({r.std()*np.sqrt(252):.2%} anual)')
print(f'  Skewness  : {r.skew():.3f}')
print(f'  Kurtosis  : {r.kurt():.3f}')
print(f'  ACF(r2) sig en {len(sig_lags)}/40 rezagos -> '
      f'{"long memory probable" if len(sig_lags) > 10 else "memoria moderada"}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Mini-EDA -- ALICORC1 BVL\n{SOURCE}', fontsize=10, y=1.01)
axes[0].plot(prices.index, prices.values, color=C['price'], lw=0.8)
axes[0].set_title('Precio ALICORC1 (S/.)'); axes[0].grid(axis='y', alpha=0.3)
col_r = np.where(r >= 0, C['green'], C['hurst'])
axes[1].bar(r.index, r.values*100, color=col_r, alpha=0.6, width=0.8)
axes[1].set_title('Retornos diarios (%)'); axes[1].axhline(0, color='black', lw=0.4)
axes[1].grid(axis='y', alpha=0.3)
axes[2].bar(range(1,41), acf_r2[1:41], color=C['dfa'], alpha=0.7, width=0.8)
axes[2].axhline(ci_acf, color=C['neutral'], lw=1.0, ls='--')
axes[2].axhline(-ci_acf, color=C['neutral'], lw=1.0, ls='--')
axes[2].axhline(0, color='black', lw=0.4)
axes[2].set_title('ACF(r^2) -- long memory?')
axes[2].set_xlabel('Rezago')
axes[2].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('data/finance_eda.png')

In [ ]:
# ── FUNCIONES R/S Y DFA ───────────────────────────────────────────────────────
def hurst_rs(series, min_n=10, max_n=None, n_points=20):
    x = np.array(series); N = len(x)
    if max_n is None: max_n = N // 2
    ns = np.unique(np.logspace(np.log10(min_n), np.log10(max_n), n_points).astype(int))
    ns = ns[ns >= min_n]
    rs_vals = []
    for n in ns:
        rs_n = []
        for start in range(0, N-n+1, n):
            sub = x[start:start+n]; m = sub.mean()
            Y = np.cumsum(sub - m); R = Y.max()-Y.min(); S = sub.std(ddof=0)
            if S > 0: rs_n.append(R/S)
        rs_vals.append(np.mean(rs_n) if rs_n else np.nan)
    ns_c = np.array(ns)[~np.isnan(rs_vals)]
    rs_c = np.array(rs_vals)[~np.isnan(rs_vals)]
    sl, ic, r, p, se = sp_stats.linregress(np.log(ns_c), np.log(rs_c))
    return sl, ic, r**2, ns_c, rs_c

def hurst_dfa(series, min_n=10, max_n=None, n_points=20):
    x = np.array(series); N = len(x)
    if max_n is None: max_n = N // 4
    Y = np.cumsum(x - x.mean())
    ns = np.unique(np.logspace(np.log10(min_n), np.log10(max_n), n_points).astype(int))
    ns = ns[ns >= min_n]
    F_vals = []
    for n in ns:
        segs = N // n
        if segs < 2: F_vals.append(np.nan); continue
        rms = []
        for s in range(segs):
            seg = Y[s*n:(s+1)*n]; t = np.arange(len(seg))
            sl2, ic2 = np.polyfit(t, seg, 1)
            rms.append(np.mean((seg-(sl2*t+ic2))**2))
        F_vals.append(np.sqrt(np.mean(rms)))
    ns_c = np.array(ns)[~np.isnan(F_vals)]
    F_c  = np.array(F_vals)[~np.isnan(F_vals)]
    F_c  = F_c[F_c > 0]; ns_c = ns_c[:len(F_c)]
    sl, ic, r, p, se = sp_stats.linregress(np.log(ns_c), np.log(F_c))
    return sl, ic, r**2, ns_c, F_c

def rolling_hurst(series, window=500, step=50):
    x = np.array(series); N = len(x)
    results, indices = [], []
    for start in range(0, N-window+1, step):
        H, _, _, _, _ = hurst_rs(x[start:start+window], n_points=12)
        results.append(H); indices.append(start + window//2)
    return np.array(results), np.array(indices)

print('Funciones definidas OK')

In [ ]:
# ── ESTIMAR H ─────────────────────────────────────────────────────────────────
r_arr = returns.values
H_rs,  ic_rs,  r2_rs,  ns_rs,  rs_vals  = hurst_rs(r_arr)
H_dfa, ic_dfa, r2_dfa, ns_dfa, dfa_vals = hurst_dfa(r_arr)
H_est = (H_rs + H_dfa) / 2

t_crit = sp_stats.t.ppf(0.975, df=len(ns_rs)-2)
se_rs  = np.sqrt((1-r2_rs)/(len(ns_rs)-2)) * np.std(np.log(ns_rs))

print('── Exponente de Hurst -- ALICORC1 BVL ──────────────────────────')
print(f'  R/S  : H = {H_rs:.4f}  R2 = {r2_rs:.4f}  '
      f'IC95% [{H_rs-t_crit*se_rs:.4f}, {H_rs+t_crit*se_rs:.4f}]')
print(f'  DFA  : H = {H_dfa:.4f}  R2 = {r2_dfa:.4f}')
print(f'  Avg  : H = {H_est:.4f}  d = {H_est-0.5:.4f}')
reg = 'PERSISTENTE' if H_est > 0.55 else ('ANTI-PERS.' if H_est < 0.45 else 'RUIDO BLANCO')
print(f'  Regimen: {reg}')

print('\nCalculando Hurst rodante...')
H_roll, idx_roll = rolling_hurst(r_arr, window=500, step=50)
dates_roll = returns.index[idx_roll]
print(f'OK: {len(H_roll)} estimaciones  H_min={H_roll.min():.3f}  H_max={H_roll.max():.3f}')

In [ ]:
# ── DASHBOARD ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 13))
fig.suptitle(f'Hurst Exponent -- ALICORC1 Bolsa de Valores de Lima\n{SOURCE}',
             fontsize=12, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 2, hspace=0.42, wspace=0.30)

ax1 = fig.add_subplot(gs[0, :])
ax1.plot(prices.index, prices.values, color=C['price'], lw=0.8)
ax1.set_ylabel('Precio (S/.)'); ax1.grid(axis='y', alpha=0.3)
ax1.set_title('Panel 1 -- ALICORC1 precio de cierre', loc='left', fontsize=10)

ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(np.log(ns_rs), np.log(rs_vals), color=C['rs'], s=40, zorder=3, label='R/S obs.')
x_f = np.linspace(np.log(ns_rs.min()), np.log(ns_rs.max()), 100)
ax2.plot(x_f, ic_rs + H_rs*x_f, color=C['hurst'], lw=1.5, label=f'H_RS={H_rs:.4f}')
ax2.plot(x_f, ic_rs + 0.5*x_f,  color=C['half'],  lw=1.0, ls='--', label='H=0.5')
ax2.set_xlabel('log(n)'); ax2.set_ylabel('log(R/S)')
ax2.set_title('Panel 2 -- R/S Analysis', loc='left', fontsize=9)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
ax3.scatter(np.log(ns_dfa), np.log(dfa_vals), color=C['dfa'], s=40, zorder=3, label='DFA obs.')
x_f2 = np.linspace(np.log(ns_dfa.min()), np.log(ns_dfa.max()), 100)
ax3.plot(x_f2, ic_dfa + H_dfa*x_f2, color=C['hurst'], lw=1.5, label=f'H_DFA={H_dfa:.4f}')
ax3.plot(x_f2, ic_dfa + 0.5*x_f2,   color=C['half'],  lw=1.0, ls='--', label='H=0.5')
ax3.set_xlabel('log(n)'); ax3.set_ylabel('log(F(n))')
ax3.set_title('Panel 3 -- DFA', loc='left', fontsize=9)
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[2, :])
ax4.fill_between(dates_roll, H_roll, 0.5,
                 where=H_roll > 0.5, alpha=0.3, color=C['hurst'], label='H>0.5 (persistente)')
ax4.fill_between(dates_roll, H_roll, 0.5,
                 where=H_roll < 0.5, alpha=0.3, color=C['dfa'],   label='H<0.5 (antipersistente)')
ax4.plot(dates_roll, H_roll, color=C['price'], lw=1.0)
ax4.axhline(0.5,   color=C['half'],  lw=1.2, ls='--', label='H=0.5')
ax4.axhline(H_est, color=C['hurst'], lw=0.8, ls=':',  label=f'H global={H_est:.4f}')
ax4.set_ylabel('H (ventana 500d)'); ax4.set_xlabel('Fecha')
ax4.set_title('Panel 4 -- Hurst rodante', loc='left', fontsize=10)
ax4.set_ylim(0.2, 0.9); ax4.legend(fontsize=8); ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('data/finance_dashboard.png')

In [ ]:

# ── TRANSFORMACIÓN Y EXPORTACIÓN ─────────────────────────────────────────────
# 1. Alinear precios con el índice de retornos (que tiene 1 día menos)
prices_aligned = prices.loc[returns.index]

# 2. Construir el DataFrame formalizando la dimensionalidad a 1D
df_alicorp = pd.DataFrame({
    'date': returns.index,
    'return': np.squeeze(returns.values),
    'price': np.squeeze(prices_aligned.values)
})

# Guardar primer CSV
df_alicorp.to_csv('data/finance_alicorc1.csv', index=False)

# 3. Exportar el Rolling Hurst (también asegurando 1D defensivamente)
df_hurst = pd.DataFrame({
    'date': dates_roll,
    'H_rolling': np.squeeze(H_roll) 
})

# Guardar segundo CSV
df_hurst.to_csv('data/finance_hurst_rolling.csv', index=False)

print('\nExportado OK')

## Conclusiones — contexto financiero

| Concepto | En ALICORC1 BVL | Implicacion |
|----------|----------------|-------------|
| **H > 0.5** | BVL menos eficiente que mercados desarrollados | Estrategias momentum validas |
| **H rodante** | Varia entre periodos — no es constante | Crisis reducen H (reversion dominante) |
| **d = H-0.5** | Grado de integracion fraccional | Parametro ARFIMA |
| **SS_Hurst** | Incertidumbre acumulada mayor con H>0.5 | Sub-proteccion con formula raiz(L) |

**Proximo:** `2_Supply_Adaptation.ipynb` — H estimado sobre errores de forecast de demanda Alicorp.